# Correcting P-vals for MWAS findings for *totiviridiae* family
### Author: Oscar Heath
### Date: Sept 12 2026

The file results.parquet contains 80K MWAS results for *totiviridiae*, with associated p-values. These raw p-values have not undergone multiple test correction. We use the Benjamin-Hochberg multiple test correction method to compute corrected p-values, with $\alpha = 0.05$, and save all significant findings.

In [1]:
import polars as pl
from statsmodels.stats.multitest import multipletests

In [2]:
df = pl.read_parquet("../data/results.parquet")
significant_results = df.filter(pl.col("pvalue") < 0.05)

In [3]:
pvals = df.select(pl.col("pvalue")).to_numpy().flatten()
rejected, corrected_pvals, _, _ = multipletests(pvals, alpha=0.05, method='fdr_bh')

In [4]:
# add a column to the dataframe with the corrected p-values
df = df.with_columns(pl.Series("corrected_pvalue", corrected_pvals))

# get the significant results
significant_results = df.filter(pl.col("corrected_pvalue") < 0.05)

In [5]:
significant_results.write_parquet("../data/significant_results.parquet")

In [6]:
# rank order findings by a combination of p-value, effect size, and number of studies

# Cohen's d: pooled-variance standardized mean difference between positive/negative groups
pooled_std = (
    (
        (pl.col("count_positive") - 1) * pl.col("std_positive") ** 2
        + (pl.col("count_negative") - 1) * pl.col("std_negative") ** 2
    )
    / (pl.col("count_positive") + pl.col("count_negative") - 2)
).sqrt()

significant_results = significant_results.with_columns(
    ((pl.col("mean_positive") - pl.col("mean_negative")) / pooled_std).alias("effect_size"),
    # proxy for "number of studies" - total samples backing the finding.
    # once findings are linked across bioprojects (see link_findings.ipynb), replace this
    # with a count of independent studies supporting the same finding.
    (pl.col("count_positive") + pl.col("count_negative")).alias("num_samples"),
)

# p-value, |effect size|, and sample count live on very different scales, so rank each
# metric (0 = least important) and average the ranks into one composite score instead of
# picking arbitrary weights. descending=True on corrected_pvalue flips the direction so
# that, for all three ranks, a higher rank means more important.
n = significant_results.height
significant_results = significant_results.with_columns(
    pl.col("corrected_pvalue").rank(method="average", descending=True).alias("pvalue_rank"),
    pl.col("effect_size").abs().rank(method="average").alias("effect_size_rank"),
    pl.col("num_samples").rank(method="average").alias("num_samples_rank"),
)

significant_results = significant_results.with_columns(
    (
        (pl.col("pvalue_rank") + pl.col("effect_size_rank") + pl.col("num_samples_rank"))
        / (3 * n)
    ).alias("importance_score")
).sort("importance_score", descending=True)

significant_results

bioproject,col_name,col_value,statistic,pvalue,mean_positive,std_positive,count_positive,mean_negative,std_negative,count_negative,corrected_pvalue,effect_size,num_samples,pvalue_rank,effect_size_rank,num_samples_rank,importance_score
str,list[str],list[str],f64,f64,f64,f64,i32,f64,f64,i32,f64,f64,i32,f64,f64,f64,f64
"""PRJNA664803""","[""genotype_attrib""]","[""MATa [HAP1+::NatMX-ACT1pr-Z3EV-ENO2term] [barcode-URA3-Z3EVpr-VMA3] [can1delta::STE2pr-SpHIS5] his3delta1 lyp1delta0 ura3delta0""]",6462.0,2.2437e-11,24.960148,12.139127,19,6.351905,4.686752,356,2.2791e-8,3.515575,375,726.0,787.0,620.0,0.878863
"""PRJNA473997""","[""source_name_attrib"", ""genotype_attrib""]","[""strain yMH951 (S288C background with repaired HAP1)"", ""MATa ura3D0 his3D200 leu2D0 lys2-128d HAP1 dsc5d""]",8247.0,4.8380e-10,24.668353,7.542944,16,5.288894,6.975192,539,3.7190e-7,2.771979,555,699.0,763.0,643.0,0.867326
"""PRJEB16729""","[""growth_condition_attrib""]","[""glucose""]",33891.0,1.0809e-24,17.233385,17.790594,155,2.116888,2.588122,274,7.0947e-21,1.389046,429,797.5,604.5,625.0,0.835187
"""PRJEB16729""","[""growth_condition_attrib""]","[""acetate""]",8579.0,1.0809e-24,2.116888,2.588122,274,17.233385,17.790594,155,7.0947e-21,-1.389046,429,797.5,604.5,625.0,0.835187
"""PRJNA577842""","[""dev_stage_attrib"", ""sex_attrib""]","[""preadult 1"", ""male""]",4508.0,3.3636e-20,0.924492,0.290744,38,0.146973,0.101317,119,1.5105e-16,4.647121,157,791.0,803.0,425.5,0.832097
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""PRJNA480220""","[""backfat_thickness_mm_attrib""]","[""11.0""]",28.0,0.000358,0.031459,0.007265,6,0.804457,1.946329,78,0.039865,-0.409849,84,44.0,188.0,272.0,0.207664
"""PRJNA587831""","[""collection_date_attrib"", ""samp_collect_device_attrib""]","[""2018-04"", ""RNA extraction""]",265.0,0.000252,0.429685,1.434824,53,0.004109,0.003262,5,0.030352,0.307802,58,103.0,143.5,189.5,0.179646
"""PRJNA587831""","[""collection_date_attrib"", ""samp_collect_device_attrib""]","[""2017-04"", ""DNA extraction""]",0.0,0.000252,0.004109,0.003262,5,0.429685,1.434824,53,0.030352,-0.307802,58,103.0,143.5,189.5,0.179646


In [7]:
# saved separately from significant_results.parquet so the embed/link pipeline
# (which reads that file fresh) is unaffected by the extra columns and new row order
significant_results.write_parquet("../data/ranked_significant_results.parquet")